# Imports

In [1]:
import sys
import torch
import os
import numpy as np
import nltk
sys.path.append('../..')
from nltk.corpus import stopwords
import pandas as pd
import plotly.express as px
from definitions import EXPERIMENTS_DIR
from typing import List, Dict, Tuple
from torch import Tensor
from src.commands.graph_learning.main import Trainer, Config, Graph
from src.commands.feature_extraction.graph_builder import EdgeTypes
from torch_geometric.data import HeteroData
from functools import cache

In [2]:
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/fellah-a/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/fellah-a/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/fellah-a/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/fellah-a/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
px.defaults.template = "seaborn"

In [4]:
def warn(*args, **kwargs):
    pass

import warnings
warnings.warn = warn

# Utils

In [5]:
def load_histories(path : str) -> pd.DataFrame:

    histories_dir = os.path.join(path, 'histories')
    histories = []

    for filename in os.listdir(histories_dir):
        filename = os.path.join(histories_dir, filename)
        df = pd.read_csv(filename)
        df['iteration'] = np.arange(len(df))
        histories.append(df)

    histories = pd.concat(histories)

    return histories #.groupby(by=['epoch', 'phase', 'config']).mean().reset_index()

In [6]:
def load_results(path : str) -> Dict[str, Dict[str, Tensor]]:

    results_dir = os.path.join(path, 'results')
    results = {}

    for filename in os.listdir(results_dir):
        result_path = os.path.join(results_dir, filename)
        exp = filename.split('.')[0].replace('predictions_', '')
        results[exp] = torch.load(result_path, map_location='cpu', weights_only=False)

    return results

In [7]:
@cache
def load_data(experiment : str) -> Tuple[Graph, HeteroData]:

    config = Config(experiment=experiment)
    trainer = Trainer(experiment=config.experiment, config=config.training_config, configs=['Attn'])

    data, graph = trainer.load_data(config=config.training_config, return_raw=True)

    return graph, data

In [8]:
@cache
def load_model(experiment : str, config_name : str) -> Tuple[Graph, HeteroData, torch.nn.Module]:

    config = Config(experiment=experiment)
    trainer = Trainer(experiment=config.experiment, config=config.training_config, configs=['Attn'])

    data, graph = load_data(experiment=experiment)

    model_config = config.training_config.configs[config_name]
    model = trainer.create_model(config=model_config, data=graph)

    print(model.load_state_dict(torch.load(model_config.model_path, weights_only=False)))

    return model

In [9]:
def get_dataframes(graph : Graph) -> List[pd.DataFrame]:

    articles_df = pd.DataFrame(data=list(map(lambda x : { 'id' : x.id,  **x.properties }, graph.get_nodes('Article').nodes_generator))).set_index('id')

    entities_df = pd.DataFrame(data=list(map(lambda x : { 'id' : x.id, **x.properties }, graph.get_nodes('Entity').nodes_generator)))
    
    triples_df = pd.DataFrame(data=list(map(lambda x : {
        "id" : x.id,
        "src" : x.src,
        "dst" : x.dst,
        "predicate" : x.properties['predicate']
    }, graph.get_edges(EdgeTypes.Entity_HASRELATIONWITH_Entity).edges_generator)))

    articles_df = articles_df \
        .join(pd.DataFrame(graph._edges[EdgeTypes.Article_HASKEYWORD_Keyword].edges_generator)[['src', 'dst']] \
        .groupby('src').agg(list)) \
        .rename(columns={'dst':'keywords'})

    return articles_df, entities_df, triples_df

In [10]:
def style_paper_plot(
    fig,
    *,
    width=950,
    height=600,
    font_family="Arial",
    font_size=14,
):
    fig.update_layout(
        width=width,
        height=height,
        font=dict(
            family=font_family,
            size=font_size,
        ),
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(
            l=80,
            r=30,
            t=30,
            b=70,
        ),
    )

    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        ticks="outside",
        showline=True,
        linewidth=1,
        linecolor="black",
    )

    fig.update_yaxes(
        showgrid=True,
        gridcolor="lightgray",
        gridwidth=0.5,
        zeroline=False,
        ticks="outside",
        showline=True,
        linewidth=1,
        linecolor="black",
    )

    return fig

In [21]:
EXPERIMENT = "topic_modeling/pooling_dd/pooling_stdnorm"
EXPERIMENT_PATH = os.path.join(EXPERIMENTS_DIR, EXPERIMENT)
config = Config(experiment=EXPERIMENT)

In [22]:
history = load_histories(EXPERIMENT_PATH)
results = load_results(EXPERIMENT_PATH)

In [23]:
graph, data = load_data(EXPERIMENT)

len(self.configs) = 1


In [24]:
aggregated_history = history.drop(
    columns=[
        'iteration', 
        'grad_min', 
        'grad_max', 
        'grad_mean'
    ],
    errors="ignore"
).groupby(by=['epoch','phase', 'config']).mean().reset_index()

In [25]:
aggregated_history.head()

,epoch,phase,config,loss,doc_rec_loss__MAE,doc_rec_loss__R2,doc_rec_loss,aux_loss
0,0,train,DMoN,0.159325,0.317970,-8.344928,0.159325,NaN
1,0,train,DMoN_matmul_norm_pool=true,0.109447,0.254870,-5.474146,0.109447,NaN
2,0,train,DMoN_restore,0.191673,0.352938,-10.326084,0.191673,NaN
3,0,train,DMoN_restore_norm_pool=true,0.106760,0.251855,-5.208947,0.106760,NaN
4,0,train,ENAD_matmul,0.456053,0.603108,-25.750597,0.456053,NaN


In [26]:
active_clusters = { 
    k :  len(v['S'][-1][:,-128:].argmax(dim=-1).unique()) if len(v['S']) >= 1 else None
    for k, v in results.items() 
}

In [27]:
summary = aggregated_history[(aggregated_history['epoch'] == 49) & (aggregated_history['phase'] == 'val')]
summary['Num. Active Clusters'] = summary['config'].apply(active_clusters.get)
summary

,epoch,phase,config,loss,doc_rec_loss__MAE,doc_rec_loss__R2,doc_rec_loss,aux_loss,Num. Active Clusters
693,49,val,DMoN,0.006746,0.060054,0.604472,0.006746,NaN,5
694,49,val,DMoN_matmul_norm_pool=true,0.006529,0.058879,0.618167,0.006529,NaN,25
695,49,val,DMoN_restore,0.006796,0.062225,0.600735,0.006796,NaN,2
696,49,val,DMoN_restore_norm_pool=true,0.002388,0.036839,0.860048,0.002388,NaN,15
697,49,val,ENAD_matmul,0.356994,0.521825,-19.999010,0.356994,NaN,2
698,49,val,ENAD_restore,0.357775,0.482063,-20.177123,0.357775,NaN,1
699,49,val,VQ,0.007254,0.064301,0.588102,0.006988,0.000532,21


In [28]:
clusters = results['DMoN_restore_norm_pool=true']['S'][-1].argmax(dim=-1)
ids = data['Entity'].og_node_id

records = {}

for node_id, cluster_id in zip(ids.tolist(), clusters.tolist()):

    entity = graph._nodes['Entity']._nodes[node_id].properties['name']

    if cluster_id in records:
        records[cluster_id]['Entities'].append(entity)
    else:
        records[cluster_id] = {}
        records[cluster_id]['Cluster ID'] = cluster_id
        records[cluster_id]['Entities'] = [entity]

df = pd.DataFrame(data=list(records.values()))
df['Entities'] = df['Entities'].apply('\n'.join)

In [29]:
df.to_excel('cluster_articles_reconstruction.xlsx')

In [19]:
fig = px.line(data_frame=aggregated_history, x='epoch', y='loss', color='config')
fig = style_paper_plot(fig=fig)
fig.update_layout(xaxis_title='Epoch', yaxis_title="Mean Squared Error (MSE)")
fig.show()
fig.write_image('loss_history.pdf')

In [20]:
fig = px.line(data_frame=aggregated_history, x='epoch', y='doc_rec_loss__MAE', color='config')
fig = style_paper_plot(fig=fig)
fig.update_layout(xaxis_title='Epoch', yaxis_title="Mean Absolute Error (MAE)")
fig.show()

ValueError: Value of 'y' is not the name of a column in 'data_frame'. Expected one of ['epoch', 'phase', 'config', 'loss', 'rec_loss__MAE', 'rec_loss__R2', 'rec_loss', 'aux_loss'] but received: doc_rec_loss__MAE

In [27]:
fig = px.line(data_frame=aggregated_history, x='epoch', y='doc_rec_loss__R2', color='config')
fig = style_paper_plot(fig=fig)
fig.update_yaxes(range=[-20, aggregated_history['doc_rec_loss__R2'].max().item()])
fig.update_layout(xaxis_title='Epoch', yaxis_title="R2 Score")
fig.show()